# 00 — Overview del sistema hospitalario

Este notebook explica la arquitectura completa de **laSalle Health Center**: servicios Docker, almacenamiento, flujo de datos, modelos de IA, dashboard, monitorización y automatización.

Úsalo para abrir la defensa del proyecto y explicar que no es un script aislado, sino una infraestructura containerizada.

In [2]:

from pathlib import Path
import json
import os
import subprocess
import textwrap
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start: Path | None = None) -> Path:
    """Busca la raíz del proyecto localizando docker-compose.yml."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "docker-compose.yml").exists():
            return candidate
    # Si el notebook se ejecuta desde notebooks/ antes de abrir el repo correctamente.
    return start


ROOT = find_project_root()
print(f"ROOT = {ROOT}")


def read_json(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}")
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_env(path: Path) -> dict:
    env = {}
    path = Path(path)
    if not path.exists():
        return env
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env


def run_cmd(cmd: str, timeout: int = 60) -> str:
    """Ejecuta un comando shell y devuelve stdout/stderr como texto."""
    print(f"$ {cmd}")
    try:
        result = subprocess.run(
            cmd,
            shell=True,
            cwd=ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        print(f"exit_code={result.returncode}")
        return (result.stdout or "") + (result.stderr or "")
    except Exception as exc:
        print(f"[error] {exc}")
        return str(exc)


def show_bar(series, title: str, ylabel: str = "valor"):
    ax = pd.Series(series).plot(kind="bar")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()



print("=== Diagnóstico general del proyecto ===")
print(f"ROOT: {ROOT}")
print(f"Existe docker-compose.yml: {(ROOT / 'docker-compose.yml').exists()}")
print(f"Existe Dockerfile: {(ROOT / 'Dockerfile').exists()}")
print(f"Existe .env.example: {(ROOT / '.env.example').exists()}")
print(f"Existe .env local: {(ROOT / '.env').exists()}")

print("\n=== Carpetas principales ===")
main_dirs = [
    "services",
    "models",
    "data",
    "monitoring",
    "init",
    "scripts",
    "specs",
    "docs",
    "notebooks",
    "COVID-19_Radiography_Dataset",
]
for d in main_dirs:
    p = ROOT / d
    print(f"{d:32} -> {'OK' if p.exists() else 'NO ENCONTRADO'}")

print("\n=== Servicios declarados en Docker Compose ===")
services_output = run_cmd("docker compose config --services", timeout=60)
services = [x.strip() for x in services_output.splitlines() if x.strip() and not x.startswith("exit_code")]
services_df = pd.DataFrame({"service": services})
display(services_df)

print("\n=== Estado actual de contenedores ===")
run_cmd("docker compose ps", timeout=60)

print("\n=== Variables relevantes del .env ===")
env = read_env(ROOT / ".env")
interesting_keys = [
    "API_PORT",
    "DASHBOARD_PORT",
    "MINIO_API_PORT",
    "MINIO_CONSOLE_PORT",
    "POSTGRES_DB",
    "MONGO_DB",
    "S3_BUCKET_RAW",
    "PIPELINE_PROCESSING_ENGINE",
    "DASK_SCHEDULER_ADDRESS",
    "ML_MODEL_PATH",
    "ML_TRIAGE_MODEL_PATH",
    "ML_DISEASE_MODEL_PATH",
    "ML_INFERENCE_URL",
]
rows = []
for k in interesting_keys:
    rows.append({"variable": k, "value": env.get(k, "NO DEFINIDA")})
display(pd.DataFrame(rows))

print("\n=== Puertos esperados de la demo ===")
ports_df = pd.DataFrame([
    {"servicio": "API", "url": f"http://localhost:{env.get('API_PORT', '8005')}", "uso": "REST API"},
    {"servicio": "Dashboard", "url": f"http://localhost:{env.get('DASHBOARD_PORT', '8502')}", "uso": "Streamlit"},
    {"servicio": "Dask Dashboard", "url": "http://localhost:8787", "uso": "Procesamiento escalable"},
    {"servicio": "MinIO API", "url": f"http://localhost:{env.get('MINIO_API_PORT', '9000')}", "uso": "S3 compatible"},
    {"servicio": "MinIO Console", "url": f"http://localhost:{env.get('MINIO_CONSOLE_PORT', '9001')}", "uso": "UI MinIO"},
    {"servicio": "pgAdmin", "url": "http://localhost:5050", "uso": "UI PostgreSQL"},
    {"servicio": "Mongo Express", "url": "http://localhost:8081", "uso": "UI MongoDB"},
    {"servicio": "Loki", "url": "http://localhost:3100", "uso": "Logs centralizados"},
])
display(ports_df)

ROOT = c:\Users\aripa\Downloads\Practica_Hospital_AriadnaPascualPolBallarin
=== Diagnóstico general del proyecto ===
ROOT: c:\Users\aripa\Downloads\Practica_Hospital_AriadnaPascualPolBallarin
Existe docker-compose.yml: True
Existe Dockerfile: True
Existe .env.example: True
Existe .env local: True

=== Carpetas principales ===
services                         -> OK
models                           -> OK
data                             -> OK
monitoring                       -> OK
init                             -> OK
scripts                          -> OK
specs                            -> OK
docs                             -> OK
notebooks                        -> OK
COVID-19_Radiography_Dataset     -> OK

=== Servicios declarados en Docker Compose ===
$ docker compose config --services
dask-scheduler
dask-worker
loki
mongodb
mongo-express
minio
postgres
pipeline
ml-inference
ml-triage
api
pgadmin
promtail
dashboard
minio-init
automation

exit_code=0


,service
0,dask-scheduler
1,dask-worker
2,loki
3,mongodb
4,mongo-express
5,minio
6,postgres
7,pipeline
8,ml-inference
9,ml-triage



=== Estado actual de contenedores ===
$ docker compose ps
NAME                                                           IMAGE                                                        COMMAND                  SERVICE          CREATED        STATUS                    PORTS
practica_hospital_ariadnapascualpolballarin-api-1              practica_hospital_ariadnapascualpolballarin-api              "uvicorn api_app.maiâ€¦"   api              45 hours ago   Up 45 hours (healthy)     0.0.0.0:8005->8000/tcp, [::]:8005->8000/tcp
practica_hospital_ariadnapascualpolballarin-automation-1       practica_hospital_ariadnapascualpolballarin-automation       "python -u main.py"      automation       27 hours ago   Up 27 hours               
practica_hospital_ariadnapascualpolballarin-dashboard-1        practica_hospital_ariadnapascualpolballarin-dashboard        "streamlit run app.pâ€¦"   dashboard        15 hours ago   Up 15 hours (healthy)     0.0.0.0:8502->8501/tcp, [::]:8502->8501/tcp
practica_hospi

,variable,value
0,API_PORT,8005
1,DASHBOARD_PORT,8502
2,MINIO_API_PORT,9000
3,MINIO_CONSOLE_PORT,9001
4,POSTGRES_DB,hospital
5,MONGO_DB,hospital
6,S3_BUCKET_RAW,hospital-raw
7,PIPELINE_PROCESSING_ENGINE,dask
8,DASK_SCHEDULER_ADDRESS,tcp://dask-scheduler:8786
9,ML_MODEL_PATH,/app/models/radiography/rx-efficientnetb0-2026...



=== Puertos esperados de la demo ===


,servicio,url,uso
0,API,http://localhost:8005,REST API
1,Dashboard,http://localhost:8502,Streamlit
2,Dask Dashboard,http://localhost:8787,Procesamiento escalable
3,MinIO API,http://localhost:9000,S3 compatible
4,MinIO Console,http://localhost:9001,UI MinIO
5,pgAdmin,http://localhost:5050,UI PostgreSQL
6,Mongo Express,http://localhost:8081,UI MongoDB
7,Loki,http://localhost:3100,Logs centralizados


## 1. Flujo lógico end-to-end

```text
Formulario / CSV / Radiografía
        ↓
Dashboard / API
        ↓
Pipeline
        ↓
PostgreSQL + MongoDB + MinIO
        ↓
Modelos IA:
  - ml-triage
  - ml-inference
        ↓
Dashboard + informes + alertas + logs
```

La idea principal es separar responsabilidades:

- el dashboard muestra y recoge datos;
- la API expone endpoints;
- el pipeline procesa;
- las bases de datos persisten;
- los modelos predicen;
- Loki/Promtail centralizan logs;
- automation crea alertas.

In [ ]:
# Comprobación de carpetas clave
paths = [
    "docker-compose.yml",
    "Dockerfile",
    "services/api",
    "services/dashboard",
    "services/pipeline",
    "services/ml-triage",
    "services/ml-inference",
    "services/automation",
    "monitoring",
    "models",
    "specs",
    "docs/diario-ia",
]

status = []
for p in paths:
    path = ROOT / p
    status.append({"ruta": p, "existe": path.exists(), "tipo": "dir" if path.is_dir() else "file" if path.is_file() else "missing"})

pd.DataFrame(status)


## 2. Servicios Docker Compose

La arquitectura se levanta con `docker compose up -d`. Cada servicio tiene una responsabilidad separada.

In [ ]:
# Lectura de docker-compose.yml para listar servicios.
# Si PyYAML no está disponible, muestra una instrucción alternativa.
compose_path = ROOT / "docker-compose.yml"
try:
    import yaml
    compose = yaml.safe_load(compose_path.read_text(encoding="utf-8"))
    rows = []
    for name, cfg in compose.get("services", {}).items():
        ports = cfg.get("ports", [])
        depends = cfg.get("depends_on", [])
        if isinstance(depends, dict):
            depends = list(depends.keys())
        rows.append({
            "servicio": name,
            "imagen/target": cfg.get("image") or (cfg.get("build") or {}).get("target"),
            "puertos": ", ".join(map(str, ports)) if ports else "interno",
            "depends_on": ", ".join(depends) if depends else "—",
        })
    services_df = pd.DataFrame(rows)
    services_df
except Exception as exc:
    print("No se pudo parsear docker-compose.yml con PyYAML:", exc)
    print("Puedes verlo con: docker compose config --services")


## 3. Capa de almacenamiento

| Tecnología | Uso | Justificación |
|---|---|---|
| PostgreSQL | Pacientes, ingresos y datos estructurados | Integridad relacional, SQL y consistencia. |
| MongoDB | Predicciones, eventos, rechazos, alertas y documentos flexibles | Esquema flexible para outputs de modelos y eventos. |
| MinIO | Landing zone/raw: CSVs, JSONs online, posibles imágenes | Patrón tipo S3 reproducible localmente. |

Esto cumple el requisito de usar más de un tipo de almacenamiento y simula una arquitectura Big Data real.

In [ ]:
storage = pd.DataFrame([
    {"capa": "Raw / landing zone", "tecnologia": "MinIO", "ejemplos": "CSV batch, JSON online, imágenes originales"},
    {"capa": "Estructurada", "tecnologia": "PostgreSQL", "ejemplos": "pacientes, ingresos"},
    {"capa": "Documental/eventos", "tecnologia": "MongoDB", "ejemplos": "predictions, system_events, ingestion_rejects, alerts"},
])
storage


## 4. Modelos de IA del proyecto

| Servicio | Modelo | Entrada | Salida |
|---|---|---|---|
| `ml-triage` | Modelo tabular de triaje | Síntomas y antecedentes | Alta / Media / Baja |
| `ml-triage` | Modelo tabular de enfermedad | Síntomas y antecedentes | Sospecha diagnóstica orientativa |
| `ml-inference` | Deep Learning radiografías | Imagen PNG/JPG | Sana / Neumonía / COVID-19 |

Los modelos están versionados en `models/`. Los pesos pesados no deben subirse a Git, pero sí las métricas, matrices y análisis crítico.

In [ ]:
# Resumen de artefactos disponibles en models/
model_rows = []
models_root = ROOT / "models"
if models_root.exists():
    for sub in sorted(models_root.iterdir()):
        if sub.is_dir():
            n_files = sum(1 for _ in sub.rglob("*") if _.is_file())
            model_rows.append({"carpeta": str(sub.relative_to(ROOT)), "ficheros": n_files})
pd.DataFrame(model_rows)


## 5. Qué enseñar en la defensa

Orden recomendado:

1. `docker compose ps`: demostrar servicios.
2. Dashboard: pacientes + triaje + radiografías.
3. Pipeline batch: seed + batch + eventos.
4. Calidad: CSV inválido con rechazos.
5. Modelos tabulares: métricas y matrices.
6. Radiografías: comparación CNN / ResNet18 / EfficientNet-B0.
7. Loki/Promtail: logging centralizado.
8. Automation: alertas en MongoDB.
9. Ética/legal: sistema de apoyo, no diagnóstico.